# Kaggle mẫu — chạy đúng một lượt ablation

Notebook mặc định chạy **A1 LayerNorm / seed 42 / đúng 6.000 bước**. Bảo chỉ sửa cell `CẤU HÌNH LƯỢT CHẠY`; không sửa file YAML hay mã model.

Quy trình an toàn: kiểm T4 + token → tải code → chuẩn bị cùng dữ liệu/tokenizer → smoke 6 bước → train/resume → chỉ chấm khi đủ 6.000 bước → đẩy kết quả lên Hugging Face.

> Lần đầu để `CHI_SMOKE = True` rồi Run All. Khi xanh hết, đổi thành `False` và Run All lại.

## 1 — CẤU HÌNH LƯỢT CHẠY (cell duy nhất cần sửa)

In [ ]:
# Mẫu đã điền: A1 LayerNorm, seed 42.
CONFIG = "configs/ablation_a1_layernorm.yaml"
SEED = 42

# Repo model/checkpoint của Bảo. HF_TOKEN phải có quyền Write vào repo này.
REPO_HUB = "mgbao/envi-nmt-scratch-transformer"
TOKENIZER_REPO = "mgbao/envi-nmt-scratch-transformer"

# Repo code và branch chứa bộ ablation mới. Sau khi merge vào main thì đổi GIT_REF = "main".
GIT_REPO = "https://github.com/Phuthanh123456/bku-project.git"
GIT_REF = "feat/ablation-baseline"

CHI_SMOKE = True       # lượt đầu True; xanh hết thì đổi False và Run All lại
GIO_TOI_DA = 10.5      # Kaggle thường cắt ở khoảng 12 giờ

# Các lượt hợp lệ để chia nhau, KHÔNG chạy improved/42 vì đã có checkpoint.
CONFIG_HOP_LE = {
    "vanilla": "configs/baseline_vanilla_architecture_6000.yaml",
    "a1": "configs/ablation_a1_layernorm.yaml",
    "a2": "configs/ablation_a2_warmup.yaml",
    "a3": "configs/ablation_a3_label_smoothing.yaml",
    "a4": "configs/ablation_a4_sincos.yaml",
    "a5": "configs/ablation_a5_relu.yaml",
    "a6": "configs/ablation_a6_post_norm.yaml",
    "improved": "configs/baseline_cai_tien_6000.yaml",
}
assert CONFIG in CONFIG_HOP_LE.values()
assert SEED in (42, 1337)
assert not (CONFIG == CONFIG_HOP_LE["improved"] and SEED == 42), \
    "improved/seed42 đã có rồi; chọn một trong 15 lượt còn thiếu"
print(f"Lượt đã chọn: {CONFIG} / seed {SEED} / smoke={CHI_SMOKE}")

## 2 — Cài thư viện và lấy đúng code từ GitHub

In [ ]:
!pip install -q tokenizers==0.21.0 sacrebleu==2.6.0 huggingface_hub==0.27.1 PyYAML==6.0.2 pandas matplotlib

import os, shutil, subprocess, sys
from pathlib import Path

# Run All lần hai: kernel còn đứng trong repo của lượt trước. Phải đi ra
# thư mục cha TRƯỚC khi xóa repo, nếu không git clone sẽ lỗi getcwd.
os.chdir("/kaggle/working")
WORK = Path("/kaggle/working/bku-project")
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--branch", GIT_REF, "--depth", "1", GIT_REPO, str(WORK)], check=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("Commit đang chạy:")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
assert Path(CONFIG).exists(), f"Branch {GIT_REF} chưa có {CONFIG}"

## 3 — Chặn sớm nếu quên GPU, Internet hoặc HF_TOKEN

Trên Kaggle: Settings → Accelerator → GPU T4; Add-ons → Secrets → tạo `HF_TOKEN` loại Write và bật công tắc cho notebook.

In [ ]:
import torch
assert torch.cuda.is_available(), "Không có CUDA: bật Accelerator GPU T4 rồi Restart session"
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))

from nmt.training.hub_sync import doc_token
from huggingface_hub import HfApi
TOKEN = doc_token(bat_buoc=True)
api = HfApi(token=TOKEN)
api.create_repo(REPO_HUB, private=True, exist_ok=True)
api.repo_info(REPO_HUB)
print("GPU, Internet và quyền Hub đều OK:", REPO_HUB)

## 4 — Chuẩn bị dữ liệu và lấy ĐÚNG tokenizer dùng chung

Không tự train tokenizer mới cho từng ablation vì token ID khác nhau sẽ làm phép so mất ý nghĩa.

In [ ]:
def chay(args):
    print("$", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), check=True)

chay([sys.executable, "scripts/prepare_data.py", "--config", "configs/base.yaml"])
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id=TOKENIZER_REPO,
    filename="artifacts/tokenizer/tokenizer.json",
    local_dir=str(WORK),
    token=TOKEN,
)
for f in ["data/processed/train.en", "data/processed/train.vi",
          "data/processed/tst2012.en", "data/processed/tst2013.en",
          "artifacts/tokenizer/tokenizer.json"]:
    p = Path(f)
    assert p.exists() and p.stat().st_size > 0, f"Thiếu {f}"
    print(f"OK {f}: {p.stat().st_size:,} bytes")

## 5 — Smoke test 6 bước

Lượt smoke có namespace riêng trên Hub nên không thể đè checkpoint thật. Khi cell này xanh, đổi `CHI_SMOKE=False` rồi Run All lại.

In [ ]:
if CHI_SMOKE:
    chay([sys.executable, "scripts/train.py", "--config", CONFIG,
          "--seed", SEED, "--repo-hub", REPO_HUB, "--smoke"])
    print("\nSMOKE ĐÃ QUA. Đổi CHI_SMOKE=False ở cell 1 rồi Run All lại.")
else:
    print("Đang ở chế độ train thật.")

## 6 — Train thật hoặc tự resume tới đúng 6.000 bước

Nếu hết phiên trước khi đủ bước, checkpoint đã lên Hub. Mở phiên mới với đúng CONFIG + SEED và Run All; `--tiep-tuc` tự kéo về.

In [ ]:
from nmt.utils import nap_config
cfg = nap_config(CONFIG)
assert cfg.huan_luyen.so_buoc_toi_da == 6000, "Ablation phải khóa đúng 6.000 bước"
RUN_NAME = f"{cfg.thi_nghiem.ten}_seed{SEED}"
CK_DIR = Path(cfg.thi_nghiem.thu_muc_checkpoint) / RUN_NAME

if not CHI_SMOKE:
    chay([sys.executable, "scripts/train.py", "--config", CONFIG,
          "--seed", SEED, "--so-buoc", 6000, "--repo-hub", REPO_HUB,
          "--tiep-tuc", "--gio-toi-da", GIO_TOI_DA])
else:
    print("Bỏ qua train thật trong lượt smoke.")

## 7 — Chỉ đánh giá khi checkpoint đã đủ ngân sách

In [ ]:
COMPLETE = False
if not CHI_SMOKE:
    latest = CK_DIR / "moi_nhat.pt"
    best = CK_DIR / "tot_nhat.pt"
    assert latest.exists(), f"Không thấy {latest}"
    meta = torch.load(latest, map_location="cpu", weights_only=False)
    current_step = int(meta["buoc"])
    COMPLETE = current_step >= 6000 and best.exists()
    print(f"Tiến độ {RUN_NAME}: {current_step}/6000 | complete={COMPLETE}")
    if COMPLETE:
        output = f"results/du_doan_test_{RUN_NAME}.csv"
        chay([sys.executable, "scripts/evaluate.py", "--config", CONFIG,
              "--seed", SEED, "--checkpoint", best, "--split", "test",
              "--search", "greedy", "--kv-cache", "--output", output])
        chay([sys.executable, "scripts/run_ablations.py"])
        chay([sys.executable, "scripts/summarize_ablations.py"])
    else:
        print("Chưa đủ 6.000 bước: mở phiên Kaggle mới và Run All lại, không chấm non.")

## 8 — Đẩy bảng kết quả và tải gói bàn giao

Checkpoint/log đã được train.py đồng bộ theo `RUN_NAME`. Cell này đẩy thêm CSV đánh giá rồi tạo zip nhỏ để gửi nhóm.

In [ ]:
if COMPLETE:
    files = [
        "results/diem_chinh.csv",
        "results/tong_hop_ablation.csv",
        "results/tong_hop_diem_ablation.csv",
        "results/tong_hop_ablation_mean_std.csv",
        f"results/du_doan_test_{RUN_NAME}.csv",
        f"results/bao_cao/{RUN_NAME}.md",
        f"results/cau_hinh/{RUN_NAME}.yaml",
    ]
    for f in files:
        p = Path(f)
        if p.exists():
            api.upload_file(
                path_or_fileobj=str(p), path_in_repo=f"ablation-results/{RUN_NAME}/{p.name}",
                repo_id=REPO_HUB, commit_message=f"kết quả {RUN_NAME}",
            )
    bundle = shutil.make_archive(f"/kaggle/working/{RUN_NAME}_ket_qua", "zip",
                                  root_dir=WORK, base_dir="results")
    print("XONG. Tải file ở panel Output:", bundle)
    print("Gửi nhóm đúng mã lượt:", RUN_NAME)
elif CHI_SMOKE:
    print("Smoke xong; chưa có kết quả thật để bàn giao.")
else:
    print("Checkpoint đang an toàn trên Hub; chạy phiên kế tiếp cho đủ 6.000 bước.")

## Bảo đổi lượt như thế nào?

Chỉ đổi `CONFIG` và `SEED` ở cell 1. Ví dụ A2 seed 1337: `CONFIG = CONFIG_HOP_LE["a2"]`, `SEED = 1337`. Giữ nguyên 6.000 bước, dữ liệu, tokenizer, Greedy + KV-cache và mọi siêu tham số khác.

Trước khi chạy, ghi tên lượt vào nhóm để không có hai người cùng đốt GPU cho một cặp CONFIG/SEED.